In [1]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [2]:
# --- 1. Universal Styling & Clean Aesthetics ---
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Calibri', 'DejaVu Serif'],
    'font.size': 24,               
    'axes.labelsize': 24,          
    'axes.titlesize': 30,          
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 16,         # Smaller legend size as requested
    'axes.grid': True,             
    'grid.alpha': 0.3,
    'savefig.dpi': 300,            
    'savefig.bbox': 'tight'        
})

In [3]:
# Lock in colors so methods match across ALL figures natively
method_colors = {
    'RefitRotate': '#7f7f7f', # gray
    'DynamicBVH':  '#ff7f0e', # orange
    'KineticBVH':  '#2ca02c', # green
    'AwareBVH':    '#1f77b4'  # blue
}

In [4]:
# --- 2. Data Loading Logic ---
MESH_RESOLUTIONS = {
    'Suzanne': [('4K', 3936), ('16K', 15744), ('63K', 62976), ('252K', 251904)],
    'Dragon':  [('3K', 3402), ('14K', 13614), ('54K', 54456), ('218K', 217826)],
    'Plane':   [('4K', 4352), ('17K', 16896), ('67K', 66560), ('264K', 264192)],
}

In [5]:
def load_data():
    files = glob.glob("benchall-*.csv")
    summary_list, frames_list = [], []
    for f in files:
        with open(f, 'r') as file:
            header = file.readline().strip('# \n')
            meta = dict(item.split('=') for item in header.split(','))
        df = pd.read_csv(f, comment='#')
        method, deformation, triangles = meta['method'], meta['deformation'], int(meta['triangles'])
        mesh_name = next((m for m in MESH_RESOLUTIONS.keys() if m.lower() in f.lower()), "Unknown")
        
        stats = df.mean(numeric_only=True).to_dict()
        stats.update(meta)
        stats['mesh'], stats['triangles'] = mesh_name, triangles
        summary_list.append(stats)
        
        df_f = df.copy()
        df_f['method'], df_f['deformation'], df_f['mesh'], df_f['triangles'] = method, deformation, mesh_name, triangles
        frames_list.append(df_f)
            
    df_stats = pd.DataFrame(summary_list)
    df_frames = pd.concat(frames_list) if frames_list else pd.DataFrame()
    
    m_order = ['RefitRotate', 'KineticBVH', 'DynamicBVH', 'AwareBVH']
    if not df_stats.empty: df_stats['method'] = pd.Categorical(df_stats['method'], categories=m_order, ordered=True)
    if not df_frames.empty: df_frames['method'] = pd.Categorical(df_frames['method'], categories=m_order, ordered=True)
        
    return df_stats, df_frames

In [6]:
df_stats, df_frames = load_data()

In [7]:
plt.rcParams.update({  
    'axes.labelsize': 30,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
})

In [8]:
# --- 3. Plot Generation (Exporting Individual PDFs) ---

# FIG 1: Scaling Curves (3 Individual PDFs)
for mesh in ['Suzanne', 'Dragon', 'Plane']:
    data = df_stats[(df_stats['mesh'] == mesh) & (df_stats['deformation'] == 'Localized')].sort_values('triangles')
    if not data.empty:
        plt.figure(figsize=(6, 5))
        sns.lineplot(data=data, x='triangles', y='updateMs', hue='method', palette=method_colors, marker='o', linewidth=3, markersize=8)
        # plt.title(f"{mesh}")
        plt.xlabel("Triangles")
        plt.ylabel("Update Time (ms)")
        plt.legend(title="")
        sns.despine()
        plt.tight_layout()
        plt.savefig(f"_new_plots\\fig1_{mesh.lower()}.pdf")
        plt.close()

In [9]:
print("\n[FIGURE 1 & 4] Scaling & Vertex Work (Localized Deformation)")
print("Shows Update Time (ms) and Vertices Checked across different triangle counts.")
print("-" * 60)
for mesh in ['Suzanne', 'Dragon', 'Plane']:
    data_m = df_stats[(df_stats['mesh'] == mesh) & (df_stats['deformation'] == 'Localized')]
    if not data_m.empty:
        summary = data_m.groupby(['triangles', 'method'])[['updateMs', 'verticesChecked']].mean().dropna()
        print(f"\n>>> MESH: {mesh}")
        print(summary.to_string())


[FIGURE 1 & 4] Scaling & Vertex Work (Localized Deformation)
Shows Update Time (ms) and Vertices Checked across different triangle counts.
------------------------------------------------------------

>>> MESH: Suzanne
                         updateMs  verticesChecked
triangles method                                  
967       RefitRotate    0.309921     2.901000e+03
          KineticBVH     1.177906     1.564468e+04
          DynamicBVH     0.125477     2.901000e+03
          AwareBVH       0.172658     4.719383e+02
15744     RefitRotate    5.529659     4.723200e+04
          KineticBVH    17.141675     2.438486e+05
          DynamicBVH     2.262883     4.723200e+04
          AwareBVH       2.321827     5.913741e+03
62976     RefitRotate   21.998530     1.889280e+05
          KineticBVH    63.217766     9.078901e+05
          DynamicBVH    12.464208     1.889280e+05
          AwareBVH       9.268091     1.862005e+04
251904    RefitRotate   73.119991     7.557120e+05
          Kinet

C:\Users\o1a2h\AppData\Local\Temp\ipykernel_7248\1668578407.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = data_m.groupby(['triangles', 'method'])[['updateMs', 'verticesChecked']].mean().dropna()
C:\Users\o1a2h\AppData\Local\Temp\ipykernel_7248\1668578407.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = data_m.groupby(['triangles', 'method'])[['updateMs', 'verticesChecked']].mean().dropna()
C:\Users\o1a2h\AppData\Local\Temp\ipykernel_7248\1668578407.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass obser

In [10]:
plt.rcParams.update({     
    'axes.labelsize': 24,   
    'xtick.labelsize': 24,  
    'ytick.labelsize': 16,     
})

In [11]:
# FIG 2: Overhead (Single PDF)
data_loc = df_stats[df_stats['deformation'] == 'Localized'].copy()
if not data_loc.empty:
    max_tris_mask = data_loc.groupby('mesh')['triangles'].transform(max)
    data_high_res = data_loc[data_loc['triangles'] == max_tris_mask]
    plt.figure(figsize=(8, 6))
    
    # Added the order parameter here
    sns.barplot(data=data_high_res, x='mesh', y='updateMs', hue='method', 
                palette=method_colors, order=['Suzanne', 'Dragon', 'Plane'])
    
    # plt.title("BVH Overhead (Highest Resolution)")
    plt.xlabel("")
    plt.ylabel("Update Time (ms)")
    plt.legend(title="")
    sns.despine()
    plt.tight_layout()
    plt.savefig("_new_plots/fig2_overhead.pdf")
    plt.close()

C:\Users\o1a2h\AppData\Local\Temp\ipykernel_7248\1121962696.py:4: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  max_tris_mask = data_loc.groupby('mesh')['triangles'].transform(max)


In [12]:
print("\n\n[FIGURE 2] BVH Overhead (Highest Resolution, Localized Deformation)")
print("Shows Update Time (ms) for the highest resolution of each mesh.")
print("-" * 60)
data_loc = df_stats[df_stats['deformation'] == 'Localized'].copy()
if not data_loc.empty:
    # Fixed the pandas FutureWarning by using string 'max' instead of built-in max
    max_tris_mask = data_loc.groupby('mesh')['triangles'].transform('max')
    data_high_res = data_loc[data_loc['triangles'] == max_tris_mask]
    summary = data_high_res.groupby(['mesh', 'method'])['updateMs'].mean().dropna()
    print(summary.to_string())



[FIGURE 2] BVH Overhead (Highest Resolution, Localized Deformation)
Shows Update Time (ms) for the highest resolution of each mesh.
------------------------------------------------------------
mesh     method     
Dragon   RefitRotate     41.530266
         KineticBVH     106.610291
         DynamicBVH      25.096345
         AwareBVH        12.867140
Plane    RefitRotate     76.941825
         KineticBVH     210.193780
         DynamicBVH      45.874130
         AwareBVH        31.094779
Suzanne  RefitRotate     73.119991
         KineticBVH     218.020072
         DynamicBVH      51.079185
         AwareBVH        35.638421


C:\Users\o1a2h\AppData\Local\Temp\ipykernel_7248\478845553.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = data_high_res.groupby(['mesh', 'method'])['updateMs'].mean().dropna()


In [13]:
# FIG 3: Locality Impact (Single PDF)
data_plane = df_stats[df_stats['mesh'] == 'Plane']
if not data_plane.empty:
    max_p = data_plane['triangles'].max()
    data_p_high = data_plane[data_plane['triangles'] == max_p]
    plt.figure(figsize=(8, 6))
    sns.barplot(data=data_p_high, x='deformation', y='updateMs', hue='method', palette=method_colors, order=['SineWave', 'Twist', 'Localized'])
    # plt.title("Deformation Locality Impact")
    plt.xlabel("")
    plt.ylabel("Update Time (ms)")
    plt.legend(title="")
    sns.despine()
    plt.tight_layout()
    plt.savefig("_new_plots\\fig3_locality.pdf")
    plt.close()

In [14]:
print("\n\n[FIGURE 3] Deformation Locality Impact (Plane, Highest Resolution)")
print("Shows Update Time (ms) across different deformation types.")
print("-" * 60)
data_plane = df_stats[df_stats['mesh'] == 'Plane']
if not data_plane.empty:
    max_p = data_plane['triangles'].max()
    data_p_high = data_plane[data_plane['triangles'] == max_p]
    summary = data_p_high.groupby(['deformation', 'method'])['updateMs'].mean().dropna()
    print(summary.to_string())



[FIGURE 3] Deformation Locality Impact (Plane, Highest Resolution)
Shows Update Time (ms) across different deformation types.
------------------------------------------------------------
deformation  method     
Localized    RefitRotate     76.941825
             KineticBVH     210.193780
             DynamicBVH      45.874130
             AwareBVH        31.094779
SineWave     RefitRotate     78.081207
             KineticBVH     238.879703
             DynamicBVH      78.469571
             AwareBVH        44.719738
Twist        RefitRotate     84.117573
             KineticBVH     141.839509
             DynamicBVH      77.398009
             AwareBVH        41.490839


C:\Users\o1a2h\AppData\Local\Temp\ipykernel_7248\2732616805.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = data_p_high.groupby(['deformation', 'method'])['updateMs'].mean().dropna()


In [15]:
plt.rcParams.update({
    'axes.labelsize': 30,   
    'xtick.labelsize': 20,  
    'ytick.labelsize': 20,    
})

In [16]:
# FIG 4: Vertex Work (3 Individual PDFs)
method_styles = {
    'RefitRotate': {'ls': '-',  'lw': 3, 'marker': 'o', 'color': method_colors['RefitRotate']},
    'DynamicBVH':  {'ls': '--', 'lw': 3, 'marker': 's', 'color': method_colors['DynamicBVH']},
    'KineticBVH':  {'ls': ':',  'lw': 3, 'marker': '^', 'color': method_colors['KineticBVH']},
    'AwareBVH':    {'ls': '-',  'lw': 3, 'marker': 'D', 'color': method_colors['AwareBVH']}
}
for mesh in ['Suzanne', 'Dragon', 'Plane']:
    data_m = df_stats[(df_stats['mesh'] == mesh) & (df_stats['deformation'] == 'Localized')].sort_values('triangles')
    if not data_m.empty:
        plt.figure(figsize=(6, 5))
        for m, style in method_styles.items():
            sub = data_m[data_m['method'] == m]
            if not sub.empty:
                plt.plot(sub['triangles'], sub['verticesChecked'], label=m, **style)
        plt.yscale('log')
        # plt.title(f"{mesh}")
        plt.xlabel("Triangles")
        plt.ylabel("Vertices Checked")
        plt.legend(title="", fontsize=12)
        sns.despine()
        plt.tight_layout()
        plt.savefig(f"_new_plots\\fig4_{mesh.lower()}.pdf")
        plt.close()

In [17]:
print("\n[FIGURE 1 & 4] Scaling & Vertex Work (Localized Deformation)")
print("Shows Update Time (ms) and Vertices Checked across different triangle counts.")
print("-" * 60)
for mesh in ['Suzanne', 'Dragon', 'Plane']:
    data_m = df_stats[(df_stats['mesh'] == mesh) & (df_stats['deformation'] == 'Localized')]
    if not data_m.empty:
        summary = data_m.groupby(['triangles', 'method'])[['updateMs', 'verticesChecked']].mean().dropna()
        print(f"\n>>> MESH: {mesh}")
        print(summary.to_string())


[FIGURE 1 & 4] Scaling & Vertex Work (Localized Deformation)
Shows Update Time (ms) and Vertices Checked across different triangle counts.
------------------------------------------------------------

>>> MESH: Suzanne
                         updateMs  verticesChecked
triangles method                                  
967       RefitRotate    0.309921     2.901000e+03
          KineticBVH     1.177906     1.564468e+04
          DynamicBVH     0.125477     2.901000e+03
          AwareBVH       0.172658     4.719383e+02
15744     RefitRotate    5.529659     4.723200e+04
          KineticBVH    17.141675     2.438486e+05
          DynamicBVH     2.262883     4.723200e+04
          AwareBVH       2.321827     5.913741e+03
62976     RefitRotate   21.998530     1.889280e+05
          KineticBVH    63.217766     9.078901e+05
          DynamicBVH    12.464208     1.889280e+05
          AwareBVH       9.268091     1.862005e+04
251904    RefitRotate   73.119991     7.557120e+05
          Kinet

C:\Users\o1a2h\AppData\Local\Temp\ipykernel_7248\1668578407.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = data_m.groupby(['triangles', 'method'])[['updateMs', 'verticesChecked']].mean().dropna()
C:\Users\o1a2h\AppData\Local\Temp\ipykernel_7248\1668578407.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = data_m.groupby(['triangles', 'method'])[['updateMs', 'verticesChecked']].mean().dropna()
C:\Users\o1a2h\AppData\Local\Temp\ipykernel_7248\1668578407.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass obser

In [18]:
# FIG 5: Timeline Stability (Single PDF)
if not df_frames.empty:
    max_d = df_frames[df_frames['mesh'] == 'Dragon']['triangles'].max()
    df_f5 = df_frames[(df_frames['mesh'] == 'Dragon') & (df_frames['triangles'] == max_d) & (df_frames['deformation'] == 'Localized')].copy()
    
    if not df_f5.empty:
        # Define interpolation function to normalize every run to exactly 0-100%
        def get_averaged_timeline(df, metric_col):
            common_x = np.linspace(0, 100, 300)
            all_y_interpolated = []
            for run_id in df['run'].unique():
                run_data = df[df['run'] == run_id].sort_values('frame')
                
                # Normalize this specific run's frames to 0-100%
                f_min, f_max = run_data['frame'].min(), run_data['frame'].max()
                x_vals = (run_data['frame'] - f_min) / (f_max - f_min) * 100
                y_vals = run_data[metric_col]
                
                all_y_interpolated.append(np.interp(common_x, x_vals, y_vals))
            return common_x, np.mean(all_y_interpolated, axis=0)

        plt.figure(figsize=(8, 6))
        
        # Plot each method using the normalized interpolation
        for m in df_stats['method'].cat.categories:
            df_m = df_f5[df_f5['method'] == m]
            if not df_m.empty:
                x, y = get_averaged_timeline(df_m, 'totalMs')
                plt.plot(x, y, linestyle='-', linewidth=3, color=method_colors.get(m, 'black'), label=m)
                
        # plt.title("Averaged Temporal Stability (Normalized)")
        plt.xlabel("Simulation Progress (%)")
        plt.ylabel("Total Time (ms)")
        plt.xlim(0, 100)
        plt.legend(title="")
        sns.despine()
        plt.tight_layout()
        plt.savefig("_new_plots/fig5_timeline.pdf")
        plt.close()

In [19]:
print("\n\n[FIGURE 5] Timeline Stability (Dragon, Highest Res, Localized)")
print("Shows Total Time (ms) sampled at 10% intervals across the simulation timeline.")
print("-" * 60)
if not df_frames.empty:
    max_d = df_frames[df_frames['mesh'] == 'Dragon']['triangles'].max()
    df_f5 = df_frames[(df_frames['mesh'] == 'Dragon') & (df_frames['triangles'] == max_d) & (df_frames['deformation'] == 'Localized')].copy()
    
    if not df_f5.empty:
        # Downsample to 11 points (0%, 10%, ..., 100%) for text readability
        common_x = np.linspace(0, 100, 11) 
        
        for m in df_stats['method'].cat.categories:
            df_m = df_f5[df_f5['method'] == m]
            if not df_m.empty:
                all_y_interpolated = []
                for run_id in df_m['run'].unique():
                    run_data = df_m[df_m['run'] == run_id].sort_values('frame')
                    f_min, f_max = run_data['frame'].min(), run_data['frame'].max()
                    x_vals = (run_data['frame'] - f_min) / (f_max - f_min) * 100
                    y_vals = run_data['totalMs']
                    all_y_interpolated.append(np.interp(common_x, x_vals, y_vals))
                
                y_mean = np.mean(all_y_interpolated, axis=0)
                print(f"\nMethod: {m} (totalMs over timeline)")
                for pct, val in zip(common_x, y_mean):
                    print(f"  {int(pct):3d}% : {val:.3f} ms")



[FIGURE 5] Timeline Stability (Dragon, Highest Res, Localized)
Shows Total Time (ms) sampled at 10% intervals across the simulation timeline.
------------------------------------------------------------

Method: RefitRotate (totalMs over timeline)
    0% : 72.406 ms
   10% : 74.157 ms
   20% : 73.424 ms
   30% : 72.765 ms
   40% : 73.487 ms
   50% : 73.771 ms
   60% : 72.598 ms
   70% : 72.936 ms
   80% : 89.181 ms
   90% : 84.742 ms
  100% : 75.011 ms

Method: KineticBVH (totalMs over timeline)
    0% : 128.609 ms
   10% : 135.262 ms
   20% : 135.028 ms
   30% : 131.632 ms
   40% : 133.008 ms
   50% : 132.590 ms
   60% : 134.075 ms
   70% : 134.198 ms
   80% : 247.687 ms
   90% : 154.493 ms
  100% : 131.600 ms

Method: DynamicBVH (totalMs over timeline)
    0% : 61.539 ms
   10% : 56.056 ms
   20% : 56.504 ms
   30% : 56.180 ms
   40% : 56.120 ms
   50% : 57.292 ms
   60% : 57.488 ms
   70% : 61.992 ms
   80% : 66.136 ms
   90% : 63.704 ms
  100% : 60.219 ms

Method: AwareBVH (total

In [20]:
plt.rcParams.update({
    'axes.labelsize': 28,   
    'xtick.labelsize': 18,  
    'ytick.labelsize': 18,    
})

In [21]:
# FIG 6: Phase Breakdown from "maybe.ipynb" (4 Individual PDFs)
def get_averaged_line(df, metric_col):
    common_x = np.linspace(0, 100, 300)
    all_y_interpolated = []
    for run_id in df['run'].unique():
        run_data = df[df['run'] == run_id].sort_values('cycleProgress')
        x_vals = run_data['cycleProgress'] * 100
        y_vals = run_data[metric_col]
        all_y_interpolated.append(np.interp(common_x, x_vals, y_vals))
    return common_x, np.mean(all_y_interpolated, axis=0)

if not df_frames.empty:
    max_d = df_frames[df_frames['mesh'] == 'Dragon']['triangles'].max()
    data_maybe = df_frames[(df_frames['mesh'] == 'Dragon') & (df_frames['triangles'] == max_d) & (df_frames['deformation'] == 'Twist')].copy()

    if not data_maybe.empty:
        # Calculate progress natively if cycleProgress doesn't strictly exist
        if 'cycleProgress' not in data_maybe.columns:
            data_maybe['cycleProgress'] = (data_maybe['frame'] - data_maybe['frame'].min()) / (data_maybe['frame'].max() - data_maybe['frame'].min())
            
        metrics = ['deformMs', 'updateMs', 'queryMs', 'totalMs']
        titles = ['Deformation Time', 'BVH Update Time', 'Collision Query Time', 'Total Time']
        
        for i, metric in enumerate(metrics):
            plt.figure(figsize=(6, 5))
            for m in df_stats['method'].cat.categories:
                df_m = data_maybe[data_maybe['method'] == m]
                if not df_m.empty:
                    x, y = get_averaged_line(df_m, metric)
                    plt.plot(x, y, linestyle='-', linewidth=2.5, color=method_colors.get(m, 'black'), label=m)
                    
            # plt.title(titles[i])
            plt.xlabel('Cycle Progress (%)')
            plt.ylabel('Average Time (ms)')
            plt.xlim(0, 100)
            plt.ylim(bottom=0)
            
            # --- LEGEND PLACEMENT LOGIC ---
            if metric not in ['updateMs', 'totalMs']:
                plt.legend(title="")
                
            sns.despine()
            plt.tight_layout()
            plt.savefig(f"_new_plots/fig6_twist_{metric.replace('Ms','')}.pdf")
            plt.close()

In [22]:
# --- FIG 6: Phase Breakdown ---
print("\n\n[FIGURE 6] Phase Breakdown (Dragon, Highest Res, Twist)")
print("Shows phase timings sampled at 10% intervals across the cycle progress.")
print("-" * 60)
if not df_frames.empty:
    max_d = df_frames[df_frames['mesh'] == 'Dragon']['triangles'].max()
    data_maybe = df_frames[(df_frames['mesh'] == 'Dragon') & (df_frames['triangles'] == max_d) & (df_frames['deformation'] == 'Twist')].copy()

    if not data_maybe.empty:
        if 'cycleProgress' not in data_maybe.columns:
            data_maybe['cycleProgress'] = (data_maybe['frame'] - data_maybe['frame'].min()) / (data_maybe['frame'].max() - data_maybe['frame'].min())
            
        metrics = ['deformMs', 'updateMs', 'queryMs', 'totalMs']
        common_x = np.linspace(0, 100, 11) # 0% to 100% in steps of 10
        
        for metric in metrics:
            print(f"\n>>> METRIC: {metric}")
            for m in df_stats['method'].cat.categories:
                df_m = data_maybe[data_maybe['method'] == m]
                if not df_m.empty:
                    all_y_interpolated = []
                    for run_id in df_m['run'].unique():
                        run_data = df_m[df_m['run'] == run_id].sort_values('cycleProgress')
                        x_vals = run_data['cycleProgress'] * 100
                        y_vals = run_data[metric]
                        all_y_interpolated.append(np.interp(common_x, x_vals, y_vals))
                    
                    y_mean = np.mean(all_y_interpolated, axis=0)
                    # Create a single line string for compactness
                    vals_str = " | ".join([f"{val:.2f}" for val in y_mean])
                    print(f"  {m.ljust(12)}: {vals_str}")
            print(f"  Timeline (%)  : " + " | ".join([f"{int(p):02d}  " for p in common_x]))



[FIGURE 6] Phase Breakdown (Dragon, Highest Res, Twist)
Shows phase timings sampled at 10% intervals across the cycle progress.
------------------------------------------------------------

>>> METRIC: deformMs
  RefitRotate : 31.80 | 31.77 | 30.96 | 31.28 | 32.55 | 31.33 | 31.67 | 29.14 | 36.12 | 35.95 | 32.90
  KineticBVH  : 31.23 | 32.36 | 32.59 | 33.02 | 33.05 | 30.84 | 34.79 | 30.23 | 34.31 | 35.67 | 33.10
  DynamicBVH  : 32.00 | 32.37 | 31.94 | 33.42 | 32.44 | 31.31 | 32.72 | 29.34 | 33.78 | 35.98 | 31.99
  AwareBVH    : 48.56 | 51.56 | 49.72 | 49.57 | 52.97 | 49.44 | 51.40 | 58.52 | 53.97 | 57.75 | 51.98
  Timeline (%)  : 00   | 10   | 20   | 30   | 40   | 50   | 60   | 70   | 80   | 90   | 100  

>>> METRIC: updateMs
  RefitRotate : 41.58 | 40.44 | 41.41 | 40.63 | 41.37 | 41.72 | 39.70 | 58.71 | 51.61 | 53.69 | 44.50
  KineticBVH  : 50.16 | 120.87 | 64.21 | 82.51 | 83.75 | 80.27 | 91.12 | 63.64 | 99.78 | 57.75 | 100.97
  DynamicBVH  : 30.22 | 33.74 | 42.31 | 40.99 | 42.18 | 4